# 3ScoringFinBERT

In [ ]:
# ==========================
# INSTALL DEPENDENCIES
# ==========================
!pip install transformers torch tqdm --quiet
import gdown

files = {
    'detikv2_2016.csv': '1ZkRSFCWHnMXbUgvcP5HiDNhF7sQle5jN',
}

for name, file_id in files.items():
    print(f'Downloading {file_id} to {name}')
    gdown.download(id=file_id, output=name, quiet=False)


# ==========================
# LOAD DATA
# ==========================
import pandas as pd
import re
import unicodedata

files = ["detikv2_2016.csv"]
df_list = [pd.read_csv(f, on_bad_lines='skip', encoding='utf-8') for f in files]
df_full = pd.concat(df_list, ignore_index=True)

print(f"Total artikel sebelum deduplikasi: {len(df_full)}")


# ==========================
# HAPUS DUPLIKAT
# ==========================
kolom_teks = "isi"

df_full['isi_normalized'] = (
    df_full[kolom_teks]
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

n_before = len(df_full)
df_full = df_full.drop_duplicates(subset='isi_normalized', keep='first').copy()
n_after = len(df_full)
print(f"Duplikat dihapus: {n_before - n_after} artikel")
print(f"Total artikel setelah deduplikasi: {n_after}")

df_full.drop(columns=['isi_normalized'], inplace=True)


# ==========================
# FILTER DATA
# ==========================

keywords = [
  "harga minyak",
  "transisi energi",
  "perubahan iklim",
  "emisi karbon",
  "tarif listrik",
  "karbon",
  "tenaga surya",
  "kementerian energi",
  "net zero emission",
  "pln",
  "nze",
  "bioenergi",
  "jaringan listrik",
  "crude oil",
  "energi hijau",
  "minyak mentah",
  "turbin angin",
  "batubara"
]

pattern = '|'.join([re.escape(k) for k in keywords])
df_filter = df_full[
    df_full[kolom_teks].str.contains(pattern, case=False, na=False)
].copy()

print(f"\nJumlah data setelah filter kata kunci: {len(df_filter)} dari {len(df_full)}")
print(f"Persentase artikel relevan: {len(df_filter)/len(df_full)*100:.2f}%")


# ==========================
# CLEANING UNTUK FINBERT
# (tidak perlu translasi karena FinBERT-Indonesia dilatih pada teks Indonesia)
# ==========================

def clean_text_bert(text):
    """
    Cleaning khusus untuk FinBERT-Indonesia.
    Tidak perlu translasi karena model sudah dilatih pada korpus bahasa Indonesia.
    Fokus pada penghapusan elemen non-substantif yang tidak memiliki nilai semantik.
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKD", text)
    text = re.sub("\\xad", "", text)

    # --- Singkatan umum berita Indonesia ---
    text = re.sub(" Tbk\\.", " Tbk ", text)
    text = re.sub(" Tbk", " Tbk ", text)
    text = re.sub(" Rp\\.", " ", text)
    text = re.sub(" Rp", " ", text)
    text = re.sub(" PT\\.", " PT ", text)
    text = re.sub(" Pt\\.", " PT ", text)
    text = re.sub(" dr\\.", " ", text)
    text = re.sub(" Dr\\.", " ", text)
    text = re.sub(" DR\\.", " ", text)
    text = re.sub(" N\\.A\\.", " ", text)
    text = re.sub(" H\\.M\\.", " HM", text)
    text = re.sub(" jl\\. ", " jalan ", text)
    text = re.sub(" Jl\\. ", " jalan ", text)
    text = re.sub(" Jln\\. ", " jalan ", text)
    text = re.sub(" jln\\. ", " jalan ", text)

    # --- URL ---
    text = re.sub(
        r'''(?i)\b((?:https?://|www\d{0,3}[.]|[a-z0-9.\-]+[.][a-z]{2,4}/)(?:[^\s()<>]+|\(([^\s()<>]+|(\([^\s()<>]+\)))*\))+(?:\(([^\s()<>]+|(\([^\s()<>]+\)))*\)|[^\s`!()\[\]{};:'".,<>?«»\u201c\u201d\u2018\u2019]))''',
        "link-website", text
    )

    # --- Pola angka & format ---
    text = re.sub(r"\. *[0-9]+\. ", ". ", text)
    text = re.sub(r": *[0-9]+\. ", ". ", text)
    text = re.sub("(?<=\\d)(?=[a-zA-Z])", " ", text)
    text = re.sub("(?<=[a-zA-Z])(?=\\d)", " ", text)
    text = re.sub(" ([a-zA-Z]{1,2})\\. ", " ", text)
    text = re.sub(r"(?<=\d)[,.](?=\d)", "", text)

    # --- Normalisasi spasi & tanda baca ---
    text = re.sub("\\*", "", text)
    text = re.sub(":", ": ", text)
    text = re.sub(r"\.(?=\S)", ". ", text)
    text = re.sub(r"\?(?=\S)", "? ", text)
    text = re.sub(r"\)(?=\S)", ") ", text)
    text = re.sub(r"\((?=\S)", " (", text)
    text = re.sub("\\. +\\.", ".", text)
    text = re.sub(" +\\? +", "? ", text)
    text = re.sub(" +,", ", ", text)
    text = re.sub(" +\\. ", ". ", text)
    text = re.sub("\\.,", ", ", text)
    text = re.sub(" {2,}", " ", text)
    text = re.sub("\\.{2,}", ".", text)
    text = re.sub("\\?{2,}", ".", text)
    text = re.sub(r'\. *\([^)]*\)[\. ]', ". ", text)
    text = re.sub(r'\. *\([^)]*\)$', ".", text)

    return text.strip()

df_filter['isi_clean'] = df_filter['isi'].apply(clean_text_bert)

# Hapus artikel yang setelah cleaning menjadi kosong atau terlalu pendek
n_before_clean = len(df_filter)
df_filter = df_filter[df_filter['isi_clean'].str.len() >= 50].copy()
print(f"\nArtikel terhapus karena terlalu pendek setelah cleaning: "
      f"{n_before_clean - len(df_filter)}")
print(f"Total artikel siap scoring: {len(df_filter)}")


# ==========================
# LOAD FINBERT INDONESIA
# ==========================
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

MODEL_NAME = "michaelmanurung/finbert-indonesia"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Gunakan GPU jika tersedia (Google Colab dengan GPU runtime)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
model.eval()

print(f"\nModel loaded: {MODEL_NAME}")
print(f"Device: {device}")
print(f"Label mapping: 0=positive, 1=neutral, 2=negative")


# ==========================
# FINBERT SENTIMENT FUNCTION
# ==========================

def finbert_sentiment_mean(text, batch_size=16):
    """
    Hitung rata-rata probabilitas sentimen FinBERT per kalimat.

    Menggunakan pendekatan Fazlija & Harder (2022):
    mean_score = pos_mean - neg_mean → rentang -1 hingga 1

    Returns dict: mean_score, mean_pos, mean_neg, mean_neu, n_sentences
    """
    # Validasi input
    if not isinstance(text, str) or not text.strip():
        return {
            'mean_score':  0.0,
            'mean_pos':    0.0,
            'mean_neg':    0.0,
            'mean_neu':    1.0,
            'n_sentences': 0
        }

    # Tokenisasi per kalimat
    sentences = sent_tokenize(text)
    if not sentences:
        sentences = [text]

    # Batching untuk efisiensi memori
    all_probs = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        try:
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512
            )
            # Pindahkan input ke device yang sama dengan model
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model(**inputs)
                probs   = torch.nn.functional.softmax(
                    outputs.logits, dim=-1
                ).cpu().numpy()

            for prob in probs:
                all_probs.append({
                    "positive": float(prob[0]),
                    "neutral":  float(prob[1]),
                    "negative": float(prob[2])
                })

        except Exception as e:
            # Jika satu batch gagal, skip batch tersebut dan lanjutkan
            # agar pipeline tidak terhenti karena satu artikel bermasalah
            print(f"  ⚠️ Batch gagal diproses: {e}")
            continue

    # Jika semua batch gagal, kembalikan nilai netral
    if not all_probs:
        return {
            'mean_score':  0.0,
            'mean_pos':    0.0,
            'mean_neg':    0.0,
            'mean_neu':    1.0,
            'n_sentences': len(sentences)
        }

    # Hitung rata-rata probabilitas per kelas
    pos_mean = sum(p["positive"] for p in all_probs) / len(all_probs)
    neg_mean = sum(p["negative"] for p in all_probs) / len(all_probs)
    neu_mean = sum(p["neutral"]  for p in all_probs) / len(all_probs)

    # Skor utama menggunakan metode Fazlija & Harder (2022)
    mean_score = pos_mean - neg_mean

    return {
        'mean_score':  float(mean_score),
        'mean_pos':    float(pos_mean),
        'mean_neg':    float(neg_mean),
        'mean_neu':    float(neu_mean),
        'n_sentences': len(sentences)
    }


# ==========================
# APPLY MODEL
# ==========================
tqdm.pandas(desc="FinBERT Scoring")

results = df_filter['isi_clean'].progress_apply(finbert_sentiment_mean)

df_filter['mean_score']  = results.apply(lambda x: x['mean_score'])
df_filter['mean_pos']    = results.apply(lambda x: x['mean_pos'])
df_filter['mean_neg']    = results.apply(lambda x: x['mean_neg'])
df_filter['mean_neu']    = results.apply(lambda x: x['mean_neu'])
df_filter['n_sentences'] = results.apply(lambda x: x['n_sentences'])

print("FinBERT Scoring selesai!")
print(df_filter[['tanggal', 'mean_score', 'mean_pos',
                  'mean_neg', 'mean_neu', 'n_sentences']].head())


# ==========================
# LABEL SENTIMENT OTOMATIS
# ==========================
q_pos = df_filter['mean_score'].quantile(0.67)
q_neg = df_filter['mean_score'].quantile(0.33)

print(f"\nThreshold positif (Q67): {q_pos:.4f}")
print(f"Threshold negatif (Q33): {q_neg:.4f}")

def label_sentiment_auto(score):
    if score >= q_pos:
        return 'positive'
    elif score <= q_neg:
        return 'negative'
    return 'neutral'

df_filter['label_sentiment'] = df_filter['mean_score'].apply(label_sentiment_auto)

print("\nDistribusi label:")
print(df_filter['label_sentiment'].value_counts())


# ==========================
# AGREGASI HARIAN
# ==========================
df_daily = (
    df_filter.groupby('tanggal')
    .agg(
        sentiment_mean=('mean_score', 'mean'),
        pos_mean=('mean_pos',   'mean'),
        neg_mean=('mean_neg',   'mean'),
        neu_mean=('mean_neu',   'mean'),
        n_artikel=('mean_score', 'count')
    )
    .reset_index()
)

print("\nAgregasi harian:")
print(df_daily.head())
print(f"\nTotal hari dengan data sentimen: {len(df_daily)}")
print(f"Rata-rata artikel per hari     : {df_daily['n_artikel'].mean():.1f}")


# ==========================
# SIMPAN HASIL
# ==========================
df_daily.to_csv('sentiment_finbert_2016.csv', index=False)
df_filter.to_csv('sentiment_finbert_full_2016.csv', index=False)

print("\nFile tersimpan:")
print("  - sentiment_finbert_2016.csv       (agregasi harian)")
print("  - sentiment_finbert_full_2016.csv  (per artikel)")

try:
    from IPython.display import FileLink, display
    display(FileLink('sentiment_finbert_2016.csv'))
    display(FileLink('sentiment_finbert_full_2016.csv'))
except:
    pass

Downloading...
From: https://drive.google.com/uc?id=1ZkRSFCWHnMXbUgvcP5HiDNhF7sQle5jN
To: /kaggle/working/detikv2_2016.csv
100%|██████████| 43.9M/43.9M [00:00<00:00, 62.9MB/s]


Total artikel sebelum deduplikasi: 22539
Duplikat dihapus: 1 artikel
Total artikel setelah deduplikasi: 22538

Jumlah data setelah filter kata kunci: 2860 dari 22538
Persentase artikel relevan: 12.69%

Artikel terhapus karena terlalu pendek setelah cleaning: 0
Total artikel siap scoring: 2860


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Model loaded: michaelmanurung/finbert-indonesia
Device: cpu
Label mapping: 0=positive, 1=neutral, 2=negative


FinBERT Scoring: 100%|██████████| 2860/2860 [2:10:10<00:00,  2.73s/it]  


FinBERT Scoring selesai!
       tanggal  mean_score  mean_pos  mean_neg  mean_neu  n_sentences
2   2016-01-01    0.080554  0.315016  0.234461  0.450523           22
3   2016-01-01   -0.127384  0.175243  0.302628  0.522129           14
8   2016-01-01    0.267836  0.470910  0.203074  0.326016           30
13  2016-01-02   -0.238354  0.187955  0.426309  0.385736           25
17  2016-01-02   -0.165225  0.189511  0.354736  0.455753           11

Threshold positif (Q67): -0.0008
Threshold negatif (Q33): -0.1936

Distribusi label:
label_sentiment
neutral     972
positive    944
negative    944
Name: count, dtype: int64

Agregasi harian:
      tanggal  sentiment_mean  pos_mean  neg_mean  neu_mean  n_artikel
0  2016-01-01        0.073669  0.320390  0.246721  0.432890          3
1  2016-01-02       -0.201789  0.188733  0.390522  0.420744          2
2  2016-01-03       -0.179624  0.223588  0.403212  0.373200          2
3  2016-01-04       -0.113271  0.242903  0.356175  0.400922         12
4  201

/kaggle/working/sentiment_finbert_2016.csv

/kaggle/working/sentiment_finbert_full_2016.csv